# Milestone 3 Lab, notebook 2 of 6: Checks in code

## Where we are

| Notebook | What you do |
|---|---|
| 1. Where we are, and the problems by hand | Recall the question, find the data problems yourself, and correct the answer key |
| **2. Checks in code** (you are here) | Write two quality checks as queries, and prove each one works |
| 3. Quarantine, and the figures before and after | Set the wrong records aside, recalculate, and confirm the totals add up |
| 4. A to-do list and a helper, by hand | Ask a model to explain the problems, then build an agent that organises its own work |
| 5. A harness works the queue | Give every problem to a ready-made agent harness, and score its reports |
| 6. The same question, with real warnings | Ask the analyst the leader's question, now that the warnings come from real checks |

Model requests in this notebook: **0**. A full run of this lab costs about 10 US cents.

## What this notebook is for

In notebook 1 you searched for problems by hand. Here each search becomes a **check**: a query that returns the bad records. Code can run a check every time the figures are calculated, and it never forgets one.

You write two checks yourself. For each one you prove two things: it **finds** a problem that is there, and it **finds nothing** when the records are clean. A check that cannot do both is not trustworthy.

There is no AI in this notebook. Checks are code, on purpose.

In [1]:
import pandas as pd
from lab_helpers import add_row_numbers, load_source_tables, run_sql, show_code, show_doc

from retail_profit_stack.quality.checks import (
    CHECKS,
    SQL_DIR,
    load_check_cases,
    number_rows,
    run_check_case,
    run_record_checks,
)
from retail_profit_stack.transform.pipeline import load_sources

## Step 1: The design

Five checks, from the **HLD**. Each has a fixed outcome, which notebook 3 is about. Here we care only about finding the bad records.

In [2]:
show_doc("HLD.md", "5. The five checks")

> **From `docs/HLD.md`: 5. The five checks**

| Check | What it looks for | Outcome when found |
|---|---|---|
| `duplicate_records` | The same record appearing more than once | Quarantine the extra copies |
| `missing_product_cost` | A product sold in a month with no cost for that month | Needs review |
| `event_out_of_order` | A refund or shipment dated before its order | Needs review |
| `refund_exceeds_line` | A refund larger than the order line it refunds | Needs review |
| `totals_do_not_match` | A total in the profit table that differs from the kept records | Needs review |

When a check finds nothing, its outcome is **pass**.

The order matters. The four record checks run first. The records are split. The transformations run on what was kept. The totals check runs last, because it needs the profit table.

## Step 2: A tiny table with a problem in it

Start small, so you can see the right answer with your own eyes. Here are three order lines. The second is a copy of the first.

In [3]:
tiny_lines = pd.DataFrame([
    {"order_item_id": "OI-1", "order_id": "ORD-1", "product_id": "P001", "quantity": 1, "unit_list_price": 40.0, "discount_amount": 4.0},
    {"order_item_id": "OI-1", "order_id": "ORD-1", "product_id": "P001", "quantity": 1, "unit_list_price": 40.0, "discount_amount": 4.0},
    {"order_item_id": "OI-2", "order_id": "ORD-2", "product_id": "P001", "quantity": 1, "unit_list_price": 40.0, "discount_amount": 4.0},
])

tiny_lines

,order_item_id,order_id,product_id,quantity,unit_list_price,discount_amount
0,OI-1,ORD-1,P001,1,40.0,4.0
1,OI-1,ORD-1,P001,1,40.0,4.0
2,OI-2,ORD-2,P001,1,40.0,4.0


Rows 1 and 2 are identical. So how can a check say *which* one to set aside? Their values cannot tell them apart.

The answer is to number the rows. `add_row_numbers` adds a `row_number` column: 1 for the first row, 2 for the second.

In [4]:
tiny_lines = add_row_numbers(tiny_lines)

tiny_lines

,order_item_id,order_id,product_id,quantity,unit_list_price,discount_amount,row_number
0,OI-1,ORD-1,P001,1,40.0,4.0,1
1,OI-1,ORD-1,P001,1,40.0,4.0,2
2,OI-2,ORD-2,P001,1,40.0,4.0,3


## Step 3: Write the first check

The check is one SQL query. Read it from the inside out:

- The inner query gives every row a `copy_number`. Rows that are equal in every column form a group (`PARTITION BY` lists those columns). Within a group the first row gets 1, the next gets 2.
- The outer query keeps the rows whose `copy_number` is above 1. Those are the extra copies.

In [5]:
DUPLICATE_LINES_SQL = """
SELECT row_number, order_item_id, product_id, copy_number
FROM (
    SELECT *,
           row_number() OVER (
               PARTITION BY order_item_id, order_id, product_id, quantity,
                            unit_list_price, discount_amount
               ORDER BY row_number
           ) AS copy_number
    FROM order_items
)
WHERE copy_number > 1
ORDER BY row_number
"""

Run it on the tiny table. `run_sql` runs a query over pandas tables, as in milestone 2. We expect exactly one row back: row 2.

In [6]:
run_sql(DUPLICATE_LINES_SQL, {"order_items": tiny_lines})

,row_number,order_item_id,product_id,copy_number
0,2,OI-1,P001,2


It found row 2, and left rows 1 and 3 alone. Row 3 has the same product and price as row 1, but a different id, so it is a different line and not a copy.

Now the other half of the proof. Remove the copy, and the check must find nothing.

In [7]:
clean_lines = add_row_numbers(tiny_lines.drop_duplicates(subset=["order_item_id"]).drop(columns=["row_number"]))

run_sql(DUPLICATE_LINES_SQL, {"order_items": clean_lines})

,row_number,order_item_id,product_id,copy_number


An empty table: nothing found. That is what **pass** looks like.

## Step 4: Write a second check

A refund larger than its order line. One line worth $72.00 (2 × $40.00, less an $8.00 discount) and two refunds: one for $36.00, one for $90.00.

In [8]:
tiny_line = add_row_numbers(pd.DataFrame([
    {"order_item_id": "OI-1", "order_id": "ORD-1", "product_id": "P001", "quantity": 2, "unit_list_price": 40.0, "discount_amount": 8.0},
]))
tiny_refunds = add_row_numbers(pd.DataFrame([
    {"refund_id": "REF-1", "order_item_id": "OI-1", "product_id": "P001", "refund_amount": 36.0},
    {"refund_id": "REF-2", "order_item_id": "OI-1", "product_id": "P001", "refund_amount": 90.0},
]))

tiny_refunds

,refund_id,order_item_id,product_id,refund_amount,row_number
0,REF-1,OI-1,P001,36.0,1
1,REF-2,OI-1,P001,90.0,2


The query has two parts. `line_values` works out each line's value, with `DISTINCT` so a copied line counts once (the trap from notebook 1). The main query returns each refund that is larger than its line's value.

In [9]:
REFUND_EXCEEDS_LINE_SQL = """
WITH line_values AS (
    SELECT DISTINCT order_item_id,
           quantity * unit_list_price - discount_amount AS line_value
    FROM order_items
)
SELECT r.row_number, r.refund_id, r.product_id, r.refund_amount, l.line_value
FROM refunds AS r JOIN line_values AS l ON l.order_item_id = r.order_item_id
WHERE r.refund_amount > l.line_value + 0.005
ORDER BY r.row_number
"""

run_sql(REFUND_EXCEEDS_LINE_SQL, {"order_items": tiny_line, "refunds": tiny_refunds})

,row_number,refund_id,product_id,refund_amount,line_value
0,2,REF-2,P001,90.0,72.0


It found the $90.00 refund and not the $36.00 one. (The `+ 0.005` allows for half a cent of rounding.)

What about a refund of exactly $72.00, the whole line? That is a full refund, which is normal. The check must not report it.

In [10]:
full_refund = add_row_numbers(pd.DataFrame([
    {"refund_id": "REF-3", "order_item_id": "OI-1", "product_id": "P001", "refund_amount": 72.0},
]))

run_sql(REFUND_EXCEEDS_LINE_SQL, {"order_items": tiny_line, "refunds": full_refund})

,row_number,refund_id,product_id,refund_amount,line_value


Nothing found. Cases at the edge, like "exactly equal", are where checks most often go wrong, so they are worth testing.

## Step 5: Your two checks on the real records

Now run both on the business's real records.

In [11]:
tables = load_source_tables()
real_lines = add_row_numbers(tables["order_items"])
real_refunds = add_row_numbers(tables["refunds"])

run_sql(DUPLICATE_LINES_SQL, {"order_items": real_lines})

,row_number,order_item_id,product_id,copy_number
0,1247,OI-01246,P001,2
1,1435,OI-01433,P002,2
2,1437,OI-01434,P002,2
3,1440,OI-01436,P002,2
4,1442,OI-01437,P002,2
5,1756,OI-01750,P001,2
6,1931,OI-01924,P001,2


Seven extra copies: three bottle lines and four flask lines. The same ones you found by hand.

In [12]:
run_sql(REFUND_EXCEEDS_LINE_SQL, {"order_items": real_lines, "refunds": real_refunds})

,row_number,refund_id,product_id,refund_amount,line_value
0,2,REF-0002,P011,60.0,25.0


And the sun hat refund.

## Step 6: The production checks

The production code has five checks. Each is described by a small record: its name, what it looks for, its outcome, and how severe a finding is.

In [13]:
pd.DataFrame([{"check": check.name, "looks for": check.looks_for, "outcome when found": check.outcome, "severity": check.severity} for check in CHECKS])

,check,looks for,outcome when found,severity
0,duplicate_records,The same record appearing more than once.,quarantine,low
1,missing_product_cost,A product sold in a month with no cost for tha...,needs_review,high
2,event_out_of_order,A refund or shipment dated before its order.,needs_review,low
3,refund_exceeds_line,A refund larger than the order line it refunds.,needs_review,medium
4,totals_do_not_match,A total in the profit table that differs from ...,needs_review,high


Each check has one SQL file. Here is the production version of the refund check. It is the query you wrote, with the output columns every check shares: the month, the product, the table, the row number, and a sentence saying what is wrong.

In [14]:
show_code((SQL_DIR / "refund_exceeds_line.sql").read_text(), "sql")

```sql
-- Check: refund_exceeds_line
-- Finds each refund that is larger than the value of the order line it refunds.
-- A line's value is quantity x list price, less its discount.

WITH line_values AS (
    SELECT DISTINCT order_item_id,
           quantity * unit_list_price - discount_amount AS line_value
    FROM order_items
)
SELECT strftime(r.refund_date, '%Y-%m') AS month,
       r.product_id,
       'refunds' AS source_table,
       r.row_number,
       'Refund of ' || printf('%.2f', r.refund_amount)
           || ' on a line worth ' || printf('%.2f', l.line_value) || '.' AS detail
FROM refunds AS r JOIN line_values AS l ON l.order_item_id = r.order_item_id
WHERE r.refund_amount > l.line_value + 0.005
```

## Step 7: Check cases

In milestone 2, hand-worked cases judged every transformation. Checks get the same treatment. `evals/m3_check_cases.json` holds tiny tables like the ones you just made, each with the rows the check **should** find.

Here is one case, in full.

In [15]:
cases = load_check_cases()

print(len(cases), "cases")
cases[1]

15 cases


{'check': 'duplicate_records',
 'name': 'one line twice',
 'inputs': {'orders': [{'order_id': 'ORD-1',
    'order_date': '2026-10-05',
    'status': 'completed'},
   {'order_id': 'ORD-2', 'order_date': '2026-10-06', 'status': 'completed'}],
  'order_items': [{'order_item_id': 'OI-1',
    'order_id': 'ORD-1',
    'product_id': 'P001',
    'quantity': 1,
    'unit_list_price': 40.0,
    'discount_amount': 0.0},
   {'order_item_id': 'OI-1',
    'order_id': 'ORD-1',
    'product_id': 'P001',
    'quantity': 1,
    'unit_list_price': 40.0,
    'discount_amount': 0.0},
   {'order_item_id': 'OI-2',
    'order_id': 'ORD-2',
    'product_id': 'P001',
    'quantity': 1,
    'unit_list_price': 40.0,
    'discount_amount': 0.0}]},
 'expected': [{'source_table': 'order_items', 'row_number': 2}]}

Run every case. The loop below takes one case at a time, runs its check on its tiny tables, and compares what the check found with what it should find.

In [16]:
rows = []
for case in cases:
    passed, found = run_check_case(case)
    rows.append({
        "check": case["check"],
        "case": case["name"],
        "should find": len(case["expected"]),
        "found": len(found),
        "correct": "yes" if passed else "NO",
    })

pd.DataFrame(rows)

,check,case,should find,found,correct
0,duplicate_records,clean records,0,0,yes
1,duplicate_records,one line twice,1,1,yes
2,duplicate_records,one line three times,2,2,yes
3,duplicate_records,two different lines that look alike,0,0,yes
4,duplicate_records,a refund twice,1,1,yes
5,missing_product_cost,cost present,0,0,yes
6,missing_product_cost,cost missing for one month,1,1,yes
7,missing_product_cost,a cancelled order needs no cost,0,0,yes
8,event_out_of_order,dates in order,0,0,yes
9,event_out_of_order,refund before order,1,1,yes


Every check has at least one case where it should find nothing, and at least one where it should find something. All are correct.

## Step 8: All the record checks, on the real records

Four of the five checks look at the source records. (The fifth compares totals after the calculation. It comes in notebook 3.)

`load_sources` reads the seven tables, as in milestone 2. `number_rows` adds the row numbers. `run_record_checks` runs the four queries and returns every bad record, one row each.

In [17]:
sources = number_rows(load_sources())
bad_records = run_record_checks(sources)

print(len(bad_records), "bad records")
bad_records.groupby(["check_name", "source_table"]).size().rename("bad records").reset_index()

26 bad records


,check_name,source_table,bad records
0,duplicate_records,order_items,7
1,duplicate_records,payment_fees,7
2,duplicate_records,refunds,1
3,duplicate_records,shipments,7
4,event_out_of_order,refunds,1
5,event_out_of_order,shipments,1
6,missing_product_cost,product_costs,1
7,refund_exceeds_line,refunds,1


Look at `event_out_of_order`. That is a problem you did **not** look for in notebook 1: a refund and a shipment dated before their own order. A check finds what it was written to find, every time. A person finds what they think of that day.

Here are those two records.

In [18]:
pd.set_option("display.max_colwidth", 100)

bad_records[bad_records["check_name"] == "event_out_of_order"]

,check_name,month,product_id,source_table,row_number,detail
23,event_out_of_order,2026-10,P003,refunds,50,"Refund dated 2026-10-19, but its order is dated 2026-10-31."
24,event_out_of_order,2026-09,P009,shipments,64,"Shipment dated 2026-09-01, but its order is dated 2026-09-02."


The duplicate check also found a refund that appears twice, as well as the copied shipments and payment fees that belong to the copied lines.

> **For the detail:** `docs/LLD.md`, section 4, lists each check, its outcome, and its severity.

## What we now know

1. A check is a query that returns the bad records. Nothing more.
2. A check must be proven both ways: it finds a planted problem, and it passes clean records.
3. Edge cases, like a refund equal to its line, need their own test.
4. The checks found 26 bad records, including several that nobody had listed.

**Next:** notebook 3. Finding a bad record is half the job. What should happen to it?